# EDSR inference, in detail

Training happens once.  Inference happens every day, on scenes far larger than a GPU
can hold, so the scene is cut into tiles and the pieces are put back together.  Four
numbers decide whether that runs at all, how fast it is, and whether the result is
correct.  This page turns each of them one at a time and measures what happens.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('lib/infer.py', 'lib/sr_models.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('infer', 'sr_models'):
    sys.modules.pop(m, None)

import imageio.v2 as imageio
from infer import *
from sr_models import load_edsr

get = lambda p, out: urllib.request.urlretrieve(f'{BASE}/{p}', out)[0]
SCENE = imageio.imread(get('dataset/test/incheon_2001.png', 'scene.png'))
SMALL = imageio.imread(get('dataset/test/incheon_600.png', 'small.png'))
net = load_edsr(get('models/14_inference/checkpoints/edsr_x2.pt', 'edsr_x2.pt'), scale=2)

## 1. The config

Everything that changes between runs lives in one object, so a run can be reproduced
by writing down six values.

| field | what it decides |
|---|---|
| `scale` | how much bigger the output is |
| `tile` | how much of the image goes into one patch |
| `overlap` | margin read around each tile and thrown away afterwards |
| `batch` | how many tiles travel to the GPU together |
| `precision` | `fp32` or `fp16` |
| `device` | `cuda` or `cpu` |

`plan` answers what a config implies before a single pixel is touched.

In [ ]:
cfg = Config(scale=2, tile=256, overlap=16, batch=4, precision='fp32')
cfg

In [ ]:
plan(*SCENE.shape[:2], cfg)

## 2. Tile size

The tile is the unit of work.  Larger tiles mean fewer of them and proportionally
less wasted margin — but each one has to fit on the GPU.

In [ ]:
show_tiles(SCENE.shape, (256, 512, 1024), cfg)

## 3. Overlap

A convolution near the edge of a tile has no neighbours to look at.  The network pads
with zeros instead, and the outermost output rows come out wrong.  Reading a margin
and discarding it replaces those rows with ones that did have neighbours.

Orange is read; darker orange is read twice.

In [ ]:
show_overlap(SCENE.shape, (0, 16, 64), cfg)

### How much margin is enough

The small scene fits on the GPU in one piece, so the tiled result can be compared
against the answer with no tiles at all.  Whatever differs is the seam.

In [ ]:
_, seams = seam_scan(SMALL, net, cfg, overlaps=(0, 4, 8, 16, 32))
show_seam(seams)

## 4. Batch

Tiles are independent, so several can go in one forward pass.  That keeps the GPU
busy — and it is also the quickest way to run out of memory.

In [ ]:
show_batch(SCENE.shape, (1, 4, 16), cfg)

## 5. What it costs

Time and peak memory, measured on the scene itself.

In [ ]:
rows = bench(SCENE, net, [replace(cfg, tile=t) for t in (128, 256, 512)])
show_bench(rows, by='tile')

In [ ]:
rows = bench(SCENE, net, [replace(cfg, batch=b) for b in (1, 4, 8)])
show_bench(rows, by='batch')

### Memory follows one number, not two

Peak memory does not track the tile size, and it does not track the batch size.  It
tracks their product — the pixels in one forward pass.  Both knobs land on the same
line, and its slope is a property of this network.

In [ ]:
law = memory_law(SCENE, net, cfg)
show_memory_law(law)

## 6. Precision

`fp16` halves what every activation costs and runs faster where tensor cores exist.
The output is not bit-identical, so the question is whether the difference shows.

In [ ]:
a = sr_tiled(SMALL, net, replace(cfg, precision='fp32'))
b = sr_tiled(SMALL, net, replace(cfg, precision='fp16'))
show_precision(a, b)

In [ ]:
for p in ('fp32', 'fp16'):
    try_run(SCENE, net, replace(cfg, precision=p, batch=8))

## 7. Running it

With the config settled, the whole scene is one call.

In [ ]:
out = sr_tiled(SCENE, net, cfg, report=True)
show_result(SCENE, out, cfg, center=(1180, 980), size=100)

## 8. When it breaks: out of memory

Each row below asks for more pixels per forward pass than the one above.  Where the
ladder stops depends on the GPU this notebook happens to be running on — a Colab T4
has about 15 GiB, so it gives up earlier than a workstation card would.

In [ ]:
for c in (replace(cfg, tile=512, batch=16),
          replace(cfg, tile=1024, overlap=64, batch=4),
          replace(cfg, tile=256, overlap=128, batch=64)):
    try_run(SCENE, net, c)

Three ways out, cheapest first: send fewer tiles per call, use smaller tiles, drop
to `fp16`.  All three shrink the same number.

In [ ]:
for c in (replace(cfg, tile=1024, overlap=64, batch=1),
          replace(cfg, tile=256, batch=4),
          replace(cfg, tile=1024, overlap=64, batch=4, precision='fp16')):
    try_run(SCENE, net, c)

## 9. When it breaks: the output is the wrong size

The margin is read in input pixels and cropped in output pixels, and the two differ
by `scale`.  Three ways to get that wrong:

| bug | what was skipped |
|---|---|
| `no_pad` | the image was never padded up to a whole number of tiles |
| `input_crop` | the margin was cropped by `overlap` instead of `overlap * scale` |
| `no_trim` | the padding was never trimmed back off the result |

Only the last one changes the size.  That makes the other two the dangerous ones: they
write a file of exactly the expected shape, filled with the wrong pixels.

In [ ]:
show_fail(SMALL, net, cfg)